# Week 3, day 2 — Worksheet 01 SOLUTIONS: long and wide   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Question 6 is the one to re-read — the wide table has a cell the long table has
no row for, and the two shapes disagree about whether that is missing or zero.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 01 — Long and wide. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")

print("shape:", orders.shape)
print("columns:", list(orders.columns))
print()
print(orders[["OrderID", "Year", "Region", "Category", "Sales"]].head())

PART A — reading the shape you have

### Question 1

`1093` rows, `8` regions, `4` years. -> a region x year table would have **32 cells**.

1,093 rows describing something that fits in 32 cells. That ratio is the
whole reason both shapes exist: the long form carries every individual
order, the wide form carries only the summary.

Neither is more correct. You cannot recover the individual orders from the
32 cells, and you cannot read the 1,093 rows side by side.

In [ ]:
print("rows:", len(orders))
print("regions:", orders["Region"].nunique())
print("years:  ", orders["Year"].nunique())
print()
print("cells in a region x year table:",
      orders["Region"].nunique() * orders["Year"].nunique())

### Question 2

One order per row, with `Region` repeating. -> **302** rows for Ontario.

The region is written out 302 times. That redundancy is what makes long
format easy to append to and easy to filter — every row is
self-describing, so you never need to look at another row to know what this
one means.

In [ ]:
print(orders[["Year", "Region", "Sales"]].head(8))
print()
print("rows for Ontario:", (orders["Region"] == "Ontario").sum())

### Question 3

Ontario `466003.99` (302 orders), West `346344.89` (232), Atlantic `236399.66` (144), Prarie `233047.01` (216), Quebec `154771.30` (82), Yukon `104765.49` (73), NWT `48353.18` (35), Nunavut `15890.71` (9).

`groupby` is long format's natural operation: it collapses many rows into
one per group without you having to know in advance how many groups there
are.

Note Nunavut has **9 orders**. Any per-order statistic for Nunavut is
computed from nine numbers, and the region still appears in every table
alongside Ontario's 302 — same visual weight, wildly different
reliability. Carrying the count next to the total is what stops that
being misleading.

In [ ]:
summary = orders.groupby("Region").agg(
    orders=("OrderID", "count"),
    total_sales=("Sales", "sum"),
).sort_values("total_sales", ascending=False).round(2)
print(summary.to_string())

PART B — the wide shape

### Question 4

`(8, 5)` -> columns `['Region', '2009', '2010', '2011', '2012']`.

Four of the five column names are **years** — they are data that has been
promoted into the schema. You cannot filter on year, group by year, or add
a fifth year without altering the table's structure.

That is the definition of untidy, and it is what almost every spreadsheet
export looks like. Worksheet 04 turns it back.

Note also `Nunavut` has two blank cells. Q6 is about what they mean.

In [ ]:
wide = pd.read_csv("data/technology_wide.csv")
print(wide.to_string(index=False))
print()
print("shape:", wide.shape)
print("columns:", list(wide.columns))

### Question 5

From the wide file `135029.86`; from the long file **`135029.87`**.

A one-penny disagreement between two correct answers to the same question.

The wide file was written by an export that rounded every cell to two
decimal places. Summing eight already-rounded numbers is not the same as
rounding the true sum — each cell lost up to half a penny and the losses
did not cancel.

This is the reshaping version of worksheet 12's float lesson from 30/08,
and it is more common in practice: the moment data passes through a
rounded export, the totals computed downstream stop reconciling with the
source. Not by much, and not never.

The rule is to aggregate from the most granular data you have, and treat a
rounded summary as a display artefact rather than a source.

In [ ]:
wide = pd.read_csv("data/technology_wide.csv")
print(wide.dtypes)
print()
tech = orders[orders["Category"] == "Technology"]
print("from wide: ", round(wide["2010"].sum(), 2))
print("from long: ", round(tech[tech["Year"] == 2010]["Sales"].sum(), 2))

### Question 6

**2** blank cells, both `Nunavut` — `2009` and `2012`. -> the long table has **0** Technology rows for either.

The blank is not corruption. Nunavut genuinely sold no Technology in those
two years, so there is nothing to record.

The two shapes disagree about how to say that. In long format the fact is
represented by the **absence of a row** — you have to know to look for it.
In wide format it becomes a **visible empty cell**, sitting in a grid that
announces the combination should have existed.

That is the strongest argument for reshaping as a checking technique:
going wide makes gaps appear that were invisible while they were merely
missing rows. Worksheet 03 does this deliberately with `unstack`.

In [ ]:
wide = pd.read_csv("data/technology_wide.csv")
print(wide.isna().sum())
print()
missing = wide[wide.isna().any(axis=1)]
print(missing.to_string(index=False))
print()
tech = orders[orders["Category"] == "Technology"]
for _, row in missing.iterrows():
    reg = row["Region"]
    for yr in ["2009", "2010", "2011", "2012"]:
        if pd.isna(row[yr]):
            n = len(tech[(tech["Region"] == reg) & (tech["Year"] == int(yr))])
            print("long rows for Technology in %s, %s: %d" % (reg, yr, n))

### Question 7

Technology occupies **30 of 32** cells. All orders together occupy **32 of 32** — no gaps at all.

The full dataset has every combination, which is why the gap only appears
once you filter to one category. That is worth knowing before you trust a
complete-looking grid: completeness at the top level says nothing about
completeness inside any slice of it.

Slice by category and you get 2 gaps. Slice by category *and* ship mode and
you would get more. Every extra dimension multiplies the cells and the data
does not multiply with it.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
present = tech.groupby(["Region", "Year"]).size()
cells = orders["Region"].nunique() * orders["Year"].nunique()
print("Technology combinations present:", len(present))
print("cells in the wide table:        ", cells)
print("difference:                     ", cells - len(present))
print()
print("all orders, for comparison:", len(orders.groupby(["Region", "Year"]).size()),
      "of", cells, "-> no gaps at all")

PART C — which shape for which job

### Question 8

12 rows — 3 categories x 4 years. -> 2009 Technology is the largest at `260450.45`.

Twelve rows, still long format: one row per observation, with the two
identifying columns spelled out.

This is the right shape to keep working with — you can filter it, join it,
append next year to it. It is the wrong shape to *read*, which is what
Q9 is about.

In [ ]:
by_cat_year = (orders.groupby(["Year", "Category"])["Sales"]
                     .sum().round(2).reset_index())
print(by_cat_year.head(8).to_string(index=False))
print()
print("rows:", len(by_cat_year))

### Question 9

Ontario `+32663.97` (`+50.3%`), NWT `+4370.90`, Quebec `+314.44`, then four declines led by West `-16577.07`. -> **Nunavut's change and pct are both `NaN`**.

Side by side, the 2009-vs-2012 comparison reads instantly. Doing this in
long format means joining the table to itself on region with two different
year filters — several lines and easy to get wrong.

Nunavut is the lesson. Both its cells are blank, so `NaN - NaN` is `NaN`
and the percentage is `NaN` too. Arithmetic on missing data propagates
rather than raising, and the region silently drops out of your comparison
while still occupying a row.

Had the export written `0` instead of blank, the change would read `0.00`
and the percentage would be a division by zero — a very different kind of
wrong. The blank is the more honest encoding, and it still needs handling.

In [ ]:
wide = pd.read_csv("data/technology_wide.csv")
comp = wide[["Region", "2009", "2012"]].copy()
comp["change"] = (comp["2012"] - comp["2009"]).round(2)
comp["pct"] = ((comp["2012"] / comp["2009"] - 1) * 100).round(1)
print(comp.sort_values("change", ascending=False).to_string(index=False))

### Question 10

`wide["2013"]` -> **raises** `KeyError: '2013'`. -> the same question in long format returns **`0` rows and no error**.

The same question, asked of the same data in two shapes, with opposite
failure modes.

In **wide** format a year is a column, so a year you do not have is a
missing column, and asking for it is a structural error that stops you.

In **long** format a year is a value, so a year you do not have is simply
an empty filter result — a perfectly valid frame with nothing in it. Sum
it and you get `0.0`; put that in a report and it reads as 'we sold
nothing in 2013' rather than 'we have no data for 2013'.

Which you prefer depends on whether an absent year is more likely to be a
bug or a fact. But it is a real difference in behaviour, and it follows
from the shape rather than from anything you wrote.

In [ ]:
wide = pd.read_csv("data/technology_wide.csv")
print("long format answer for 2013:",
      len(orders[orders["Year"] == 2013]), "rows, no error")
print("wide columns:", list(wide.columns))
print(wide["2013"])